In [0]:
# Databricks notebook source
# GOLD LAYER: Race Performance Metrics
# Purpose: Detailed per-race, per-driver performance analysis
# Source: f1_data.silver.fact_results

from pyspark.sql.functions import (
    col, when, lit, rank, dense_rank, lag, lead,
    sum as spark_sum, avg, count, round as spark_round,
    current_timestamp, max as spark_max, min as spark_min,
    percent_rank, ntile, row_number
)
from pyspark.sql.window import Window

# ============ 1. READ FROM SILVER ============
fact = spark.read.table("f1_data.silver.fact_results")

# ============ 2. ALL SESSION TYPES (not just Race) ============
# Race performance tracks Practice, Qualifying, Race separately
# Useful for showing qualifying → race position delta

# Filter to meaningful session types
sessions_of_interest = fact.filter(
    col("session_type").isin("Race", "Sprint", "Qualifying", "Sprint Qualifying")
)

# ============ 3. RACE-SPECIFIC METRICS ============
race_only = sessions_of_interest.filter(
    col("session_type").isin("Race", "Sprint")
)

# Position gained/lost vs grid (qualifying vs race)
# First get qualifying position per driver per meeting
quali_positions = fact.filter(
    col("session_type").isin("Qualifying", "Sprint Qualifying")
).select(
    "driver_number",
    "meeting_key",
    "year",
    col("position").alias("qualifying_position")
)

# Join qualifying position to race result
race_with_quali = race_only.join(
    quali_positions,
    on=["driver_number", "meeting_key", "year"],
    how="left"
)

# ============ 4. COMPUTE PERFORMANCE METRICS ============
race_window = Window.partitionBy("meeting_key", "session_key") \
    .orderBy(col("points").desc(), col("position").asc())

race_performance = race_with_quali \
    .withColumn("positions_gained",
        # Positive = gained positions, Negative = lost positions
        when(
            col("qualifying_position").isNotNull() & col("position").isNotNull(),
            col("qualifying_position") - col("position")
        ).otherwise(lit(None))
    ) \
    .withColumn("performance_category",
        # Classify driver's race performance
        when(col("position") == 1, lit("WINNER"))
        .when(col("position") <= 3, lit("PODIUM"))
        .when(col("position") <= 10, lit("POINTS_FINISH"))
        .when(col("dnf") == True, lit("DNF"))
        .when(col("dns") == True, lit("DNS"))
        .when(col("dsq") == True, lit("DSQ"))
        .when(col("position").isNotNull(), lit("NO_POINTS"))
        .otherwise(lit("UNKNOWN"))
    ) \
    .withColumn("is_points_finish",
        when(col("points") > 0, lit(True)).otherwise(lit(False))
    ) \
    .withColumn("finishing_percentile",
        # What percentile did driver finish in (100 = winner, 0 = last)
        when(
            col("position").isNotNull(),
            spark_round(
                (1 - (col("position") - 1) /
                spark_max("position").over(Window.partitionBy("meeting_key", "session_key")))
                * 100, 1
            )
        ).otherwise(lit(None))
    ) \
    .withColumn("gap_to_winner_seconds",
        # gap_to_leader_seconds is already vs winner for top finishers
        when(col("gap_type") == "TIME_GAP", col("gap_to_leader_seconds"))
        .when(col("gap_type") == "LEADER", lit(0.0))
        .otherwise(lit(None))
    ) \
    .withColumn("lapping_efficiency",
        # Points earned per lap completed
        when(
            col("number_of_laps").isNotNull() & (col("number_of_laps") > 0),
            spark_round(col("points") / col("number_of_laps"), 4)
        ).otherwise(lit(0.0))
    )

# ============ 5. HEAD-TO-HEAD TEAMMATE COMPARISON ============
# For each race: who beat their teammate?
teammate_window = Window.partitionBy("team_name", "meeting_key", "session_key") \
    .orderBy("position")

race_performance = race_performance \
    .withColumn("team_position",
        # Rank within team: 1 = top driver of that team this race
        rank().over(teammate_window)
    ) \
    .withColumn("beat_teammate",
        when(col("team_position") == 1, lit(True)).otherwise(lit(False))
    )

# ============ 6. SEASON TREND: Was driver improving or declining? ============
season_trend_window = Window.partitionBy("driver_number", "year") \
    .orderBy("session_date")

race_performance = race_performance \
    .withColumn("prev_race_position",
        lag("position", 1).over(season_trend_window)
    ) \
    .withColumn("prev_race_points",
        lag("points", 1).over(season_trend_window)
    ) \
    .withColumn("position_trend",
        when(
            col("prev_race_position").isNotNull() & col("position").isNotNull(),
            col("prev_race_position") - col("position")
        ).otherwise(lit(None))
        # Positive = improved vs previous race
    ) \
    .withColumn("points_trend",
        when(col("prev_race_points").isNotNull(),
            col("points") - col("prev_race_points")
        ).otherwise(lit(None))
    )

# ============ 7. WRITE TO GOLD ============
race_performance_final = race_performance.select(
    # Identity
    "driver_number",
    "driver_name",
    "driver_acronym",
    "team_name",
    "team_colour",
    "driver_country_code",
    "year",

    # Race context
    "meeting_key",
    "session_key",
    "meeting_name",
    "meeting_official_name",
    "circuit_short_name",
    "circuit_type",
    "race_country",
    "race_location",
    "session_type",
    "session_date",

    # Race result
    "position",
    "qualifying_position",
    "positions_gained",
    "points",
    "number_of_laps",
    "duration",
    "dnf",
    "dns",
    "dsq",
    "gap_to_leader_raw",
    "gap_to_winner_seconds",
    "gap_type",

    # Derived metrics
    "performance_category",
    "is_points_finish",
    "finishing_percentile",
    "lapping_efficiency",
    "team_position",
    "beat_teammate",

    # Trend metrics
    "prev_race_position",
    "prev_race_points",
    "position_trend",
    "points_trend",

    # Metadata
    lit(current_timestamp()).alias("gold_processed_at"),
    lit("f1_data.silver.fact_results").alias("gold_source")
)

race_performance_final.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .partitionBy("year", "session_type") \
    .saveAsTable("f1_data.gold.race_performance")

print(f"✓ gold.race_performance written: {race_performance_final.count()} records")

✓ gold.race_performance written: 419 records


In [0]:
display(sessions_of_interest)

driver_number,meeting_key,session_key,driver_name,driver_first_name,driver_last_name,driver_acronym,team_name,team_colour,driver_country_code,session_name,session_type,session_date,year,meeting_name,meeting_official_name,circuit_short_name,circuit_type,race_country,race_location,position,points,duration,number_of_laps,dnf,dns,dsq,gap_to_leader_raw,gap_to_leader_seconds,gap_type,finished_race,on_podium,points_per_lap,driver_found_in_dim,silver_source,silver_processed_at,data_quality_flag
1,1229,9472,Max Verstappen,Max,Verstappen,VER,Red Bull Racing,3671C6,NED,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,1.0,26.0,5504.742,57,false,false,false,0,null,TIME_GAP,true,true,0.4561,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
11,1229,9472,Sergio Perez,Sergio,Perez,PER,Red Bull Racing,3671C6,MEX,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,2.0,18.0,5527.199,57,false,false,false,22.457,22.457,TIME_GAP,true,true,0.3158,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
55,1229,9472,Carlos Sainz,Carlos,Sainz,SAI,Ferrari,E8002D,ESP,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,3.0,15.0,5529.852,57,false,false,false,25.11,25.11,TIME_GAP,true,true,0.2632,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
16,1229,9472,Charles Leclerc,Charles,Leclerc,LEC,Ferrari,E8002D,MON,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,4.0,12.0,5544.411,57,false,false,false,39.669,39.669,TIME_GAP,true,false,0.2105,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
63,1229,9472,George Russell,George,Russell,RUS,Mercedes,27F4D2,GBR,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,5.0,10.0,5551.53,57,false,false,false,46.788,46.788,TIME_GAP,true,false,0.1754,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
4,1229,9472,Lando Norris,Lando,Norris,NOR,McLaren,FF8000,GBR,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,6.0,8.0,5553.2,57,false,false,false,48.458,48.458,TIME_GAP,true,false,0.1404,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
44,1229,9472,Lewis Hamilton,Lewis,Hamilton,HAM,Mercedes,27F4D2,GBR,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,7.0,6.0,5555.066,57,false,false,false,50.324,50.324,TIME_GAP,true,false,0.1053,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
81,1229,9472,Oscar Piastri,Oscar,Piastri,PIA,McLaren,FF8000,AUS,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,8.0,4.0,5560.824,57,false,false,false,56.082,56.082,TIME_GAP,true,false,0.0702,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
14,1229,9472,Fernando Alonso,Fernando,Alonso,ALO,Aston Martin,229971,ESP,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,9.0,2.0,5579.629,57,false,false,false,74.887,74.887,TIME_GAP,true,false,0.0351,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED
18,1229,9472,Lance Stroll,Lance,Stroll,STR,Aston Martin,229971,CAN,Race,Race,2024-03-02T15:00:00.000Z,2024,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,10.0,1.0,5597.958,57,false,false,false,93.216,93.216,TIME_GAP,true,false,0.0175,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED


In [0]:
display(quali_positions)

driver_number,meeting_key,year,qualifying_position


In [0]:
display(race_with_quali)

driver_number,meeting_key,year,session_key,driver_name,driver_first_name,driver_last_name,driver_acronym,team_name,team_colour,driver_country_code,session_name,session_type,session_date,meeting_name,meeting_official_name,circuit_short_name,circuit_type,race_country,race_location,position,points,duration,number_of_laps,dnf,dns,dsq,gap_to_leader_raw,gap_to_leader_seconds,gap_type,finished_race,on_podium,points_per_lap,driver_found_in_dim,silver_source,silver_processed_at,data_quality_flag,qualifying_position
1,1229,2024,9472,Max Verstappen,Max,Verstappen,VER,Red Bull Racing,3671C6,NED,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,1.0,26.0,5504.742,57,false,false,false,0,null,TIME_GAP,true,true,0.4561,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
11,1229,2024,9472,Sergio Perez,Sergio,Perez,PER,Red Bull Racing,3671C6,MEX,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,2.0,18.0,5527.199,57,false,false,false,22.457,22.457,TIME_GAP,true,true,0.3158,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
55,1229,2024,9472,Carlos Sainz,Carlos,Sainz,SAI,Ferrari,E8002D,ESP,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,3.0,15.0,5529.852,57,false,false,false,25.11,25.11,TIME_GAP,true,true,0.2632,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
16,1229,2024,9472,Charles Leclerc,Charles,Leclerc,LEC,Ferrari,E8002D,MON,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,4.0,12.0,5544.411,57,false,false,false,39.669,39.669,TIME_GAP,true,false,0.2105,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
63,1229,2024,9472,George Russell,George,Russell,RUS,Mercedes,27F4D2,GBR,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,5.0,10.0,5551.53,57,false,false,false,46.788,46.788,TIME_GAP,true,false,0.1754,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
4,1229,2024,9472,Lando Norris,Lando,Norris,NOR,McLaren,FF8000,GBR,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,6.0,8.0,5553.2,57,false,false,false,48.458,48.458,TIME_GAP,true,false,0.1404,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
44,1229,2024,9472,Lewis Hamilton,Lewis,Hamilton,HAM,Mercedes,27F4D2,GBR,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,7.0,6.0,5555.066,57,false,false,false,50.324,50.324,TIME_GAP,true,false,0.1053,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
81,1229,2024,9472,Oscar Piastri,Oscar,Piastri,PIA,McLaren,FF8000,AUS,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,8.0,4.0,5560.824,57,false,false,false,56.082,56.082,TIME_GAP,true,false,0.0702,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
14,1229,2024,9472,Fernando Alonso,Fernando,Alonso,ALO,Aston Martin,229971,ESP,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,9.0,2.0,5579.629,57,false,false,false,74.887,74.887,TIME_GAP,true,false,0.0351,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null
18,1229,2024,9472,Lance Stroll,Lance,Stroll,STR,Aston Martin,229971,CAN,Race,Race,2024-03-02T15:00:00.000Z,Bahrain Grand Prix,FORMULA 1 GULF AIR BAHRAIN GRAND PRIX 2024,Sakhir,PERMANENT,Bahrain,Sakhir,10.0,1.0,5597.958,57,false,false,false,93.216,93.216,TIME_GAP,true,false,0.0175,true,f1_data.bronze.results,2026-07-03T09:43:41.959Z,PASSED,null


In [0]:
%sql
DESCRIBE TABLE f1_data.gold.race_performance

col_name,data_type,comment
driver_number,int,null
driver_name,string,null
driver_acronym,string,null
team_name,string,null
team_colour,string,null
driver_country_code,string,null
year,int,null
meeting_key,int,null
session_key,int,null
meeting_name,string,null
